# nb_gold_pib — Camada Gold PIB (Fabric)

**Fontes:** `silver_pib` · `silver_pib_componentes`  
**Saída:** `gold_pib_municipios`  
**Granularidade:** município × ano  
**Indicadores:** PIB total · per capita · ranking · % VAB por setor

In [ ]:
%run ./nb_utils_ibge

In [ ]:
from pyspark.sql.functions import (
    col, round as spark_round, dense_rank, lag, lit, coalesce, when
)
from pyspark.sql.window import Window

## 1. Carregar Silver e montar mapeamento de clusters

In [ ]:
# Tabela auxiliar id_municipio → cluster (derivada do nb_utils_ibge)
cluster_rows = [
    (code, cluster)
    for cluster, codes in CLUSTERS.items()
    for code in codes
]
df_cluster = spark.createDataFrame(cluster_rows, ["id_municipio", "cluster"])

# PIB total e per capita — pivot de indicadores para colunas
df_pib = spark.sql("""
    SELECT
        id_municipio,
        nome_municipio,
        CAST(ano AS INT) AS ano,
        MAX(CASE WHEN indicador = 'pib_total_r_mil'  THEN valor END) AS pib_total_r_mil,
        MAX(CASE WHEN indicador = 'pib_per_capita_r' THEN valor END) AS pib_per_capita_r
    FROM silver_pib
    GROUP BY id_municipio, nome_municipio, ano
""")

# Componentes VAB — pivot para colunas
df_comp = spark.sql("""
    SELECT
        id_municipio,
        CAST(ano AS INT) AS ano,
        MAX(CASE WHEN indicador = 'vab_agropecuaria_r_mil'  THEN valor END) AS vab_agropecuaria_r_mil,
        MAX(CASE WHEN indicador = 'vab_industria_r_mil'     THEN valor END) AS vab_industria_r_mil,
        MAX(CASE WHEN indicador = 'vab_servicos_r_mil'      THEN valor END) AS vab_servicos_r_mil,
        MAX(CASE WHEN indicador = 'vab_adm_publica_r_mil'   THEN valor END) AS vab_adm_publica_r_mil,
        MAX(CASE WHEN indicador = 'impostos_liquidos_r_mil' THEN valor END) AS impostos_liquidos_r_mil
    FROM silver_pib_componentes
    GROUP BY id_municipio, ano
""")

print(f"[OK] silver_pib: {df_pib.count()} registros")
print(f"[OK] silver_pib_componentes: {df_comp.count()} registros")

## 2. Construir Gold — join + ranking + % VAB

In [ ]:
# Join PIB + Componentes + Cluster
df_gold = (
    df_pib
    .join(df_comp,    on=["id_municipio", "ano"], how="left")
    .join(df_cluster, on="id_municipio",          how="left")
)

# Ranking por ano dentro do conjunto dos 15 municípios (PIB total decrescente)
w_rank = Window.partitionBy("ano").orderBy(col("pib_total_r_mil").desc())
df_gold = df_gold.withColumn("ranking_pib", dense_rank().over(w_rank))

# Participação percentual de cada componente no PIB total
def pct(num, den):
    return spark_round(when(col(den) > 0, col(num) / col(den) * 100), 2)

df_gold = (
    df_gold
    .withColumn("pct_vab_agropecuaria", pct("vab_agropecuaria_r_mil",  "pib_total_r_mil"))
    .withColumn("pct_vab_industria",    pct("vab_industria_r_mil",     "pib_total_r_mil"))
    .withColumn("pct_vab_servicos",     pct("vab_servicos_r_mil",      "pib_total_r_mil"))
    .withColumn("pct_vab_adm_publica", pct("vab_adm_publica_r_mil",   "pib_total_r_mil"))
    .withColumn("pct_impostos",         pct("impostos_liquidos_r_mil", "pib_total_r_mil"))
)

# Schema final — desnormalizado para Direct Lake
df_gold = df_gold.select(
    "id_municipio", "nome_municipio", "cluster", "ano",
    "pib_total_r_mil", "pib_per_capita_r", "ranking_pib",
    "vab_agropecuaria_r_mil", "vab_industria_r_mil",
    "vab_servicos_r_mil", "vab_adm_publica_r_mil", "impostos_liquidos_r_mil",
    "pct_vab_agropecuaria", "pct_vab_industria",
    "pct_vab_servicos", "pct_vab_adm_publica", "pct_impostos"
)

assert df_gold.count() > 0, "[ERRO] gold_pib_municipios vazio antes de gravar"
print(f"[OK] gold_pib_municipios: {df_gold.count()} registros")
display(df_gold.orderBy("ano", "ranking_pib").limit(20))

## 3. Gravar Gold

In [ ]:
save_delta(df_gold, "gold_pib_municipios")
print("[OK] gold_pib_municipios gravada")

# Spot check — Santos, Osasco, Mauá no ano mais recente
ano_max = df_gold.agg({"ano": "max"}).collect()[0][0]
display(
    df_gold
    .filter((col("id_municipio").isin(3548500, 3534401, 3529401)) & (col("ano") == ano_max))
    .select("nome_municipio", "cluster", "ano", "pib_total_r_mil",
            "pib_per_capita_r", "ranking_pib",
            "pct_vab_industria", "pct_vab_servicos")
    .orderBy("ranking_pib")
)